# Consolidacao das saidas do LLM por interseccao

Este notebook percorre as subpastas de `dados/llm`, carrega as listas JSON produzidas para cada interseccao, consolida os registros e salva um unico CSV com todos os artigos relevantes.

In [19]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
llm_root = repo_root / 'dados' / 'llm'

pastas_interseccao = sorted([p for p in llm_root.iterdir() if p.is_dir()]) if llm_root.exists() else []

print(f'Pasta raiz de leitura: {llm_root}')
print(f'Total de pastas de interseccao encontradas: {len(pastas_interseccao)}')
display(pd.DataFrame({'interseccao': [p.name for p in pastas_interseccao], 'pasta': [str(p.relative_to(repo_root)) for p in pastas_interseccao]}))

Pasta raiz de leitura: c:\Users\ultra\OneDrive\Documentos\github\revisor\dados\llm
Total de pastas de interseccao encontradas: 2


,interseccao,pasta
0,ag,dados\llm\ag
1,ca,dados\llm\ca


In [20]:
arquivos = []
for pasta in pastas_interseccao:
    for arquivo in sorted([p for p in pasta.rglob('*') if p.is_file()]):
        arquivos.append({
            'interseccao_pasta': pasta.name.upper(),
            'arquivo': arquivo,
            'arquivo_relativo': str(arquivo.relative_to(repo_root))
        })

print(f'Total de arquivos encontrados: {len(arquivos)}')
display(pd.DataFrame([{k: v for k, v in item.items() if k != 'arquivo'} for item in arquivos]))

Total de arquivos encontrados: 7


,interseccao_pasta,arquivo_relativo
0,AG,dados\llm\ag\ag_lote_001.txt
1,CA,dados\llm\ca\ca_lote_001.txt
2,CA,dados\llm\ca\ca_lote_002.txt
3,CA,dados\llm\ca\ca_lote_003.txt
4,CA,dados\llm\ca\ca_lote_004.txt
5,CA,dados\llm\ca\ca_lote_005.txt
6,CA,dados\llm\ca\ca_lote_006.txt


In [21]:
def carregar_json_lista(caminho):
    texto = caminho.read_text(encoding='utf-8-sig')
    try:
        dados = json.loads(texto)
    except json.JSONDecodeError:
        texto_limpo = texto.strip()
        inicio = texto_limpo.find('[')
        fim = texto_limpo.rfind(']')
        if inicio == -1 or fim == -1:
            raise ValueError(f'Nao foi possivel localizar uma lista JSON em {caminho}')
        dados = json.loads(texto_limpo[inicio:fim + 1])

    if not isinstance(dados, list):
        raise ValueError(f'O arquivo {caminho} nao contem uma lista JSON no nivel raiz.')

    return dados

registros = []
falhas = []

for item_arquivo in arquivos:
    caminho = item_arquivo['arquivo']
    interseccao_pasta = item_arquivo['interseccao_pasta']
    arquivo_relativo = item_arquivo['arquivo_relativo']

    try:
        dados = carregar_json_lista(caminho)
        for item in dados:
            if isinstance(item, dict):
                registro = item.copy()
                registro['source_file'] = arquivo_relativo
                registro['source_intersection_folder'] = interseccao_pasta
                registros.append(registro)
            else:
                falhas.append({'arquivo': arquivo_relativo, 'erro': 'Item da lista nao e um objeto JSON'})
    except Exception as exc:
        falhas.append({'arquivo': arquivo_relativo, 'erro': str(exc)})

print(f'Registros carregados: {len(registros)}')
print(f'Arquivos com falha: {len(falhas)}')
if falhas:
    display(pd.DataFrame(falhas))

Registros carregados: 230
Arquivos com falha: 1


,arquivo,erro
0,dados\llm\ca\ca_lote_006.txt,Nao foi possivel localizar uma lista JSON em c:\Users\ultra\OneDrive\Documentos\github\revisor\dados\llm\ca\ca_lote_...


In [22]:
df_llm = pd.DataFrame(registros)

if not df_llm.empty:
    colunas_ordenacao = [col for col in ['intersection', 'source_intersection_folder', 'article_id', 'source_file'] if col in df_llm.columns]
    if colunas_ordenacao:
        df_llm = df_llm.sort_values(colunas_ordenacao, na_position='last').reset_index(drop=True)

print(f'Dimensoes do dataset consolidado: {df_llm.shape}')
display(df_llm.head(10))

Dimensoes do dataset consolidado: (230, 15)


,article_id,title,journal,journal_impact_factor,doi,relevance,intersection,summary_en,summary_pt,relevant_excerpts_en,relevant_excerpts_pt,justification,confidence,source_file,source_intersection_folder
0,AG_0001,Better alone than in bad company: Addressing the risks of companion chatbots through data protection by design,Computer Law and Security Review,NaN,10.1016/j.clsr.2024.106019,relevant,AG,This article examines the governance and legal control of companion AI agents (chatbots). It focuses on how regulato...,Este artigo examina a governança e o controle legal de agentes de IA de companhia (chatbots). O foco reside em como ...,"[Addressing the risks of companion chatbots through data protection by design, GDPR... already provides a solid grou...","[Abordando os riscos de chatbots de companhia por meio da proteção de dados desde o projeto, O GDPR... já fornece um...","The article discusses the governance, control, and risk mitigation of autonomous AI agents, specifically focusing on...",0.9,dados\llm\ag\ag_lote_001.txt,AG
1,CA_0021,Humanlike AI for Corporate Social Responsibility Communication: How Perceived Anthropomorphism Shapes Stakeholder Ac...,Corporate Social Responsibility and Environmental Management,5.957,10.1002/csr.70494,relevant,CA,This study examines how consumers react to anthropomorphic chatbots in the context of Corporate Social Responsibilit...,Este estudo examina como os consumidores reagem a chatbots antropomórficos no contexto da comunicação de Responsabil...,"[examine how perceived anthropomorphic chatbots, with communication style, animacy, and intelligence, are associated...","[examinar como os chatbots antropomórficos percebidos, com estilo de comunicação, animação e inteligência, estão ass...","The article analyzes consumer perception, trust, and acceptance of AI chatbots in a service-related context (CSR com...",1.0,dados\llm\ca\ca_lote_001.txt,CA
2,CA_0025,The rise of chatbots: The effect of using chatbot agents on consumers' responses to request rejection,Journal of Consumer Psychology,NaN,10.1002/jcpy.1330,relevant,CA,This research investigates how consumers perceive and evaluate service robots versus human agents when a request is ...,Esta pesquisa investiga como os consumidores percebem e avaliam robôs versus agentes humanos quando um pedido de ser...,[investigates consumers’ perceptions and evaluations of robot service agents compared with human service agents when...,[investiga as percepções e avaliações dos consumidores sobre agentes de serviço robóticos em comparação com agentes ...,"The article directly addresses consumer perception, expectations, and evaluation of AI agents in a service context.",1.0,dados\llm\ca\ca_lote_001.txt,CA
3,CA_0026,Chatbots and mental health: Insights into the safety of generative AI,Journal of Consumer Psychology,NaN,10.1002/jcpy.1393,relevant,CA,"The study explores consumer interactions with companion AI, specifically focusing on mental health contexts. It eval...","O estudo explora as interações dos consumidores com IAs de companhia, focando especificamente em contextos de saúde ...","[experiment testing consumer reaction to risky and unhelpful chatbot responses, consumers display negative reactions...","[experimento testando a reação do consumidor a respostas arriscadas e inúteis de chatbots, os consumidores apresenta...","The study focuses on consumer reaction and interpretation of AI behavior/safety, fitting the criteria for consumer i...",1.0,dados\llm\ca\ca_lote_001.txt,CA
4,CA_0027,Avoiding embarrassment online: Response to and inferences about chatbots when purchases activate self-presentation c...,Journal of Consumer Psychology,NaN,10.1002/jcpy.1414,relevant,CA,This paper investigates how self-presentation concerns influence consumer interactions with chatbots. It finds that ...,Este artigo investiga como as preocupações com a autopresentação influenciam as interações dos consumidores com chat...,[consumers feel less embarrassed with a chatbot than

In [23]:
def valor_hashavel(x):
    if isinstance(x, list):
        return tuple(valor_hashavel(i) for i in x)
    if isinstance(x, dict):
        return tuple(sorted((k, valor_hashavel(v)) for k, v in x.items()))
    return x

valores_unicos = df_llm.apply(lambda col: col.map(valor_hashavel).nunique(dropna=True)) if not df_llm.empty else pd.Series(dtype='int64')

resumo = pd.DataFrame({
    'tipo': df_llm.dtypes.astype(str),
    'nao_nulos': df_llm.notna().sum(),
    'nulos': df_llm.isna().sum(),
    'percentual_nulos': (df_llm.isna().mean() * 100).round(2),
    'valores_unicos': valores_unicos
}).sort_index() if not df_llm.empty else pd.DataFrame()

display(resumo)

,tipo,nao_nulos,nulos,percentual_nulos,valores_unicos
article_id,object,230,0,0.00,230
confidence,float64,230,0,0.00,13
doi,object,229,1,0.43,229
intersection,object,55,175,76.09,2
journal,object,230,0,0.00,94
journal_impact_factor,float64,17,213,92.61,7
justification,object,228,2,0.87,198
relevance,object,230,0,0.00,2
relevant_excerpts_en,object,221,9,3.91,56
relevant_excerpts_pt,object,221,9,3.91,56


In [24]:
if 'relevance' in df_llm.columns:
    display(df_llm['relevance'].value_counts(dropna=False).rename_axis('relevance').to_frame('quantidade'))

if 'intersection' in df_llm.columns:
    display(df_llm['intersection'].value_counts(dropna=False).rename_axis('intersection').to_frame('quantidade'))

if 'source_intersection_folder' in df_llm.columns:
    display(df_llm['source_intersection_folder'].value_counts(dropna=False).rename_axis('source_intersection_folder').to_frame('quantidade'))

if 'source_file' in df_llm.columns:
    display(df_llm['source_file'].value_counts(dropna=False).rename_axis('source_file').to_frame('quantidade').head(20))

,quantidade
relevance,
not_relevant,175
relevant,55


,quantidade
intersection,
None,175
CA,54
AG,1


,quantidade
source_intersection_folder,
CA,220
AG,10


,quantidade
source_file,
dados\llm\ca\ca_lote_001.txt,50
dados\llm\ca\ca_lote_004.txt,50
dados\llm\ca\ca_lote_003.txt,50
dados\llm\ca\ca_lote_005.txt,50
dados\llm\ca\ca_lote_002.txt,20
dados\llm\ag\ag_lote_001.txt,10


In [25]:
if 'article_id' in df_llm.columns:
    duplicados = df_llm[df_llm.duplicated(subset=['article_id'], keep=False)].sort_values('article_id')
    print(f'Registros com article_id duplicado: {len(duplicados)}')
    display(duplicados.head(20))
else:
    print('A coluna article_id nao foi encontrada no dataset consolidado.')

Registros com article_id duplicado: 0


,article_id,title,journal,journal_impact_factor,doi,relevance,intersection,summary_en,summary_pt,relevant_excerpts_en,relevant_excerpts_pt,justification,confidence,source_file,source_intersection_folder


In [26]:
if 'relevance' not in df_llm.columns:
    raise ValueError('A coluna relevance nao foi encontrada no dataset consolidado.')

df_relevantes = df_llm[df_llm['relevance'].astype(str).str.lower().eq('relevant')].copy()

if not df_relevantes.empty:
    colunas_ordenacao = [col for col in ['intersection', 'source_intersection_folder', 'article_id'] if col in df_relevantes.columns]
    if colunas_ordenacao:
        df_relevantes = df_relevantes.sort_values(colunas_ordenacao).reset_index(drop=True)

print(f'Total de artigos relevantes: {len(df_relevantes)}')
display(df_relevantes.head(10))

Total de artigos relevantes: 55


,article_id,title,journal,journal_impact_factor,doi,relevance,intersection,summary_en,summary_pt,relevant_excerpts_en,relevant_excerpts_pt,justification,confidence,source_file,source_intersection_folder
0,AG_0001,Better alone than in bad company: Addressing the risks of companion chatbots through data protection by design,Computer Law and Security Review,NaN,10.1016/j.clsr.2024.106019,relevant,AG,This article examines the governance and legal control of companion AI agents (chatbots). It focuses on how regulato...,Este artigo examina a governança e o controle legal de agentes de IA de companhia (chatbots). O foco reside em como ...,"[Addressing the risks of companion chatbots through data protection by design, GDPR... already provides a solid grou...","[Abordando os riscos de chatbots de companhia por meio da proteção de dados desde o projeto, O GDPR... já fornece um...","The article discusses the governance, control, and risk mitigation of autonomous AI agents, specifically focusing on...",0.9,dados\llm\ag\ag_lote_001.txt,AG
1,CA_0021,Humanlike AI for Corporate Social Responsibility Communication: How Perceived Anthropomorphism Shapes Stakeholder Ac...,Corporate Social Responsibility and Environmental Management,5.957,10.1002/csr.70494,relevant,CA,This study examines how consumers react to anthropomorphic chatbots in the context of Corporate Social Responsibilit...,Este estudo examina como os consumidores reagem a chatbots antropomórficos no contexto da comunicação de Responsabil...,"[examine how perceived anthropomorphic chatbots, with communication style, animacy, and intelligence, are associated...","[examinar como os chatbots antropomórficos percebidos, com estilo de comunicação, animação e inteligência, estão ass...","The article analyzes consumer perception, trust, and acceptance of AI chatbots in a service-related context (CSR com...",1.0,dados\llm\ca\ca_lote_001.txt,CA
2,CA_0025,The rise of chatbots: The effect of using chatbot agents on consumers' responses to request rejection,Journal of Consumer Psychology,NaN,10.1002/jcpy.1330,relevant,CA,This research investigates how consumers perceive and evaluate service robots versus human agents when a request is ...,Esta pesquisa investiga como os consumidores percebem e avaliam robôs versus agentes humanos quando um pedido de ser...,[investigates consumers’ perceptions and evaluations of robot service agents compared with human service agents when...,[investiga as percepções e avaliações dos consumidores sobre agentes de serviço robóticos em comparação com agentes ...,"The article directly addresses consumer perception, expectations, and evaluation of AI agents in a service context.",1.0,dados\llm\ca\ca_lote_001.txt,CA
3,CA_0026,Chatbots and mental health: Insights into the safety of generative AI,Journal of Consumer Psychology,NaN,10.1002/jcpy.1393,relevant,CA,"The study explores consumer interactions with companion AI, specifically focusing on mental health contexts. It eval...","O estudo explora as interações dos consumidores com IAs de companhia, focando especificamente em contextos de saúde ...","[experiment testing consumer reaction to risky and unhelpful chatbot responses, consumers display negative reactions...","[experimento testando a reação do consumidor a respostas arriscadas e inúteis de chatbots, os consumidores apresenta...","The study focuses on consumer reaction and interpretation of AI behavior/safety, fitting the criteria for consumer i...",1.0,dados\llm\ca\ca_lote_001.txt,CA
4,CA_0027,Avoiding embarrassment online: Response to and inferences about chatbots when purchases activate self-presentation c...,Journal of Consumer Psychology,NaN,10.1002/jcpy.1414,relevant,CA,This paper investigates how self-presentation concerns influence consumer interactions with chatbots. It finds that ...,Este artigo investiga como as preocupações com a autopresentação influenciam as interações dos consumidores com chat...,[consumers feel less embarrassed with a chatbot than

In [27]:
saida_dir = repo_root / 'dados' / 'consolidados'
saida_dir.mkdir(parents=True, exist_ok=True)

csv_path = saida_dir / 'llm_relevantes.csv'
df_relevantes.to_csv(csv_path, index=False, encoding='utf-8-sig')

print(f'CSV salvo em: {csv_path}')

CSV salvo em: c:\Users\ultra\OneDrive\Documentos\github\revisor\dados\consolidados\llm_relevantes.csv
